## Cell 0: Install

In [1]:
!pip install -q "transformers==4.57.1" "peft==0.18.0" bitsandbytes accelerate "pandas==2.2.2" "pillow==11.3.0" tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.4 MB/s eta 0:00:00


## Cell 1: Download Data

In [2]:
import kagglehub
from pathlib import Path
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
COMPETITION_NAME = "pixels-to-predictions"
COMP_DIR = Path(kagglehub.competition_download(COMPETITION_NAME))
print("Downloaded to:", COMP_DIR)

100%|██████████| 358M/358M [00:03<00:00, 123MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/competitions/pixels-to-predictions


## Cell 2: Imports & Config

In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json, re, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForVision2Seq, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
IMG_SIZE = 224
CHOICE_LETTERS = "ABCDEFGHIJ"

# Training
BATCH_SIZE_TRAIN = 1
GRAD_ACCUM_STEPS = 16
LR = 3e-5
EPOCHS = 2
WARMUP_RATIO = 0.05

# LoRA (~1.1M params)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Inference
BATCH_SIZE_INFER = 4

SAVE_DIR = Path("/content/lora_v2")

# Google Drive for checkpoints
from google.colab import drive
drive.mount("/content/drive")
DRIVE_SAVE = Path("/content/drive/MyDrive/lora_v2_best")

print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
GPU: True
  Tesla T4


## Cell 3: Load Data

In [5]:
DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists():
    DATA_DIR = COMP_DIR

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

example_name = Path(val_df.iloc[0]["image_path"]).name
matches = list(COMP_DIR.rglob(example_name))
DATA_ROOT = matches[0].parents[2]
print("DATA_ROOT:", DATA_ROOT)

Train: 3109 | Val: 1048 | Test: 1008
DATA_ROOT: /root/.cache/kagglehub/competitions/pixels-to-predictions/images


## Cell 4: Prompt + Image Helpers

In [6]:
def clean_text(x):
    if pd.isna(x): return ""
    return str(x).strip()

def build_prompt(row, include_answer=False):
    choices = row["choices"]
    choices_text = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    prompt = "<image>\n"
    prompt += "You are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint:    prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer:"

    if include_answer:
        ans = int(row["answer"])
        letter = CHOICE_LETTERS[ans]
        prompt += f" {letter}. {choices[ans]}"

    return prompt

def load_image(row):
    path = DATA_ROOT / row["image_path"]
    return Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)

# Test
row = train_df.iloc[0]
p = build_prompt(row, include_answer=True)
print(p[-100:])

 the male's tadpoles will become adult frogs

Answer: C. the male's tadpoles will become adult frogs


## Cell 5: Load v2 checkpoint

In [7]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)
model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/lora_v2_best")
model.eval()
print("Loaded v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Loaded v2


## Cell 6: Multiple prompt variants

In [10]:
def get_candidate_token_ids(tokenizer):
    cids = {}
    for letter in CHOICE_LETTERS:
        forms = [letter, " "+letter, "\n"+letter, letter+".", " "+letter+"."]
        ids = set()
        for f in forms:
            enc = tokenizer.encode(f, add_special_tokens=False)
            if enc: ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids

def build_prompt_v1(row):
    """Original prompt (used during training)"""
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    prompt = "<image>\n"
    prompt += "You are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint:    prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    return prompt

def build_prompt_v2(row):
    """Shorter — no system instruction"""
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    prompt = "<image>\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint:    prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nThe correct answer is:"
    return prompt

def build_prompt_v3(row):
    """Minimal — just question and choices"""
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))

    prompt = f"<image>\n{question}\n\n{choices_text}\n\nAnswer:"
    return prompt

def build_prompt_v4(row):
    """No lecture/hint — they might confuse more than help"""
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))

    prompt = "<image>\n"
    prompt += "Look at the image and answer the question.\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    return prompt

## Cell 7: Test all variants on val

In [11]:
candidate_ids = get_candidate_token_ids(processor.tokenizer)

def eval_prompt_variant(prompt_fn, name, eval_df):
    preds = []
    for start in range(0, len(eval_df), BATCH_SIZE_INFER):
        batch = eval_df.iloc[start:start+BATCH_SIZE_INFER]
        images = [load_image(row) for _, row in batch.iterrows()]
        prompts = [prompt_fn(row) for _, row in batch.iterrows()]

        inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

        with torch.inference_mode():
            logits = model(**inputs).logits[:, -1, :]
        log_probs = torch.log_softmax(logits, dim=-1)

        for i, (_, row) in enumerate(batch.iterrows()):
            scores = []
            for ci in range(len(row["choices"])):
                tids = candidate_ids[CHOICE_LETTERS[ci]]
                scores.append(log_probs[i, tids].max().item())
            preds.append(int(np.argmax(scores)))
        torch.cuda.empty_cache()

    correct = sum(p == a for p, a in zip(preds, eval_df["answer"]))
    acc = correct / len(eval_df)
    print(f"  {name}: {acc:.4f} ({correct}/{len(eval_df)})")
    return preds, acc

model.eval()
gc.collect(); torch.cuda.empty_cache()

N_VAL = 200
eval_df = val_df.sample(n=min(N_VAL, len(val_df)), random_state=SEED).reset_index(drop=True)

print("Prompt variant comparison:")
results = {}
for name, fn in [("v1_original", build_prompt_v1),
                  ("v2_no_system", build_prompt_v2),
                  ("v3_minimal", build_prompt_v3),
                  ("v4_no_lecture", build_prompt_v4)]:
    preds, acc = eval_prompt_variant(fn, name, eval_df)
    results[name] = (preds, acc)

best_name = max(results, key=lambda k: results[k][1])
print(f"\nBest: {best_name} ({results[best_name][1]:.4f})")

Prompt variant comparison:
  v1_original: 0.6700 (134/200)
  v2_no_system: 0.6650 (133/200)
  v3_minimal: 0.5050 (101/200)
  v4_no_lecture: 0.5500 (110/200)

Best: v1_original (0.6700)


## Cell 8: Submit best variant

In [12]:
best_fn = {"v1_original": build_prompt_v1, "v2_no_system": build_prompt_v2,
           "v3_minimal": build_prompt_v3, "v4_no_lecture": build_prompt_v4}[best_name]
print(f"Submitting with: {best_name}")

all_preds = []
for start in tqdm(range(0, len(test_df), BATCH_SIZE_INFER), desc="Test"):
    batch = test_df.iloc[start:start+BATCH_SIZE_INFER]
    images = [load_image(row) for _, row in batch.iterrows()]
    prompts = [best_fn(row) for _, row in batch.iterrows()]

    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]
    log_probs = torch.log_softmax(logits, dim=-1)

    for i, (_, row) in enumerate(batch.iterrows()):
        scores = []
        for ci in range(len(row["choices"])):
            tids = candidate_ids[CHOICE_LETTERS[ci]]
            scores.append(log_probs[i, tids].max().item())
        all_preds.append(int(np.argmax(scores)))
    torch.cuda.empty_cache()

submission = pd.DataFrame({"id": test_df["id"], "answer": all_preds})
submission.to_csv("/content/submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/submission.csv", index=False)
print(f"Saved ({len(submission)} rows)")

from google.colab import files
files.download("/content/submission.csv")

Submitting with: v1_original


Test:   0%|          | 0/252 [00:00<?, ?it/s]

KeyboardInterrupt: 